<a href="https://colab.research.google.com/github/jacoaji02/speech-ai-model-learning/blob/main/transformer_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
from transformers import AutoTokenizer, AutoModel

# --- Exercise 1: Load pre-trained BERT assets ---
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

# --- Exercise 2: Tokenize 5 custom sentences ---
sentences = [
    "I love natural language processing.",
    "Transformers revolutionized artificial intelligence.",
    "BERT reads text bidirectionally.",
    "Python is great for deep learning.",
    "AI is awesome."
]

print("--- Exercise 2: Tokenization Inspection ---")
for i, sentence in enumerate(sentences, 1):
    # 1. Convert text to token strings
    tokens = tokenizer.tokenize(sentence)

    # 2. Convert text to numerical IDs
    token_ids = tokenizer.convert_tokens_to_ids(tokens)

    print(f"\nSentence {i}: '{sentence}'")
    print(f"  └─ Tokens:   {tokens}")
    print(f"  └─ Token IDs: {token_ids}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


--- Exercise 2: Tokenization Inspection ---

Sentence 1: 'I love natural language processing.'
  └─ Tokens:   ['i', 'love', 'natural', 'language', 'processing', '.']
  └─ Token IDs: [1045, 2293, 3019, 2653, 6364, 1012]

Sentence 2: 'Transformers revolutionized artificial intelligence.'
  └─ Tokens:   ['transformers', 'revolution', '##ized', 'artificial', 'intelligence', '.']
  └─ Token IDs: [19081, 4329, 3550, 7976, 4454, 1012]

Sentence 3: 'BERT reads text bidirectionally.'
  └─ Tokens:   ['bert', 'reads', 'text', 'bid', '##ire', '##ction', '##ally', '.']
  └─ Token IDs: [14324, 9631, 3793, 7226, 7442, 7542, 3973, 1012]

Sentence 4: 'Python is great for deep learning.'
  └─ Tokens:   ['python', 'is', 'great', 'for', 'deep', 'learning', '.']
  └─ Token IDs: [18750, 2003, 2307, 2005, 2784, 4083, 1012]

Sentence 5: 'AI is awesome.'
  └─ Tokens:   ['ai', 'is', 'awesome', '.']
  └─ Token IDs: [9932, 2003, 12476, 1012]


In [2]:
# Prepare a single tokenized batch optimized for PyTorch ('pt')
# padding=True forces shorter sentences to match the longest one by adding [PAD] tokens
inputs = tokenizer(sentences, padding=True, return_tensors="pt")

# Forward pass through BERT without calculating gradients (saves RAM)
model.eval()
with torch.no_grad():
    outputs = model(**inputs)

# Extract the final hidden state layers
last_hidden_state = outputs.last_hidden_state

print("\n--- Exercise 3: Hidden State Diagnostics ---")
print(f"Full Hidden State Shape [Batch, Seq_Len, Hidden_Dim]: {last_hidden_state.shape}")

# Grab the first token embedding (the special [CLS] token) of the first sentence
first_token_embedding = last_hidden_state[0, 0, :]
print(f"First Token Vector Slice (First 5 numbers out of 768) :\n{first_token_embedding[:5]}")


--- Exercise 3: Hidden State Diagnostics ---
Full Hidden State Shape [Batch, Seq_Len, Hidden_Dim]: torch.Size([5, 10, 768])
First Token Vector Slice (First 5 numbers out of 768) :
tensor([-0.0419,  0.0434, -0.2534, -0.3502, -0.3743])


In [3]:
short_txt = "Hello world."
long_txt = "The quick brown fox jumps over the lazy dog while checking its deep neural network parameters."

inputs_short = tokenizer(short_txt, return_tensors="pt")
inputs_long = tokenizer(long_txt, return_tensors="pt")

with torch.no_grad():
    out_short = model(**inputs_short).last_hidden_state
    out_long = model(**inputs_long).last_hidden_state

print("\n--- Exercise 4: Structural Scale Comparison ---")
print(f"Short Sentence Text Output Shape: {out_short.shape}")
# Expected output sequence length dimension will be small (e.g., [1, 4, 768])

print(f"Long Sentence Text Output Shape : {out_long.shape}")
# Expected output sequence length dimension scales way higher (e.g., [1, 18, 768])


--- Exercise 4: Structural Scale Comparison ---
Short Sentence Text Output Shape: torch.Size([1, 5, 768])
Long Sentence Text Output Shape : torch.Size([1, 19, 768])
